# 🎯 ChuckleNet: AST Laughter Labeler (Pre-trained AudioSet)

**Uses MIT/ast-finetuned-audioset-10-10-0.4593** (1.2M downloads)
- Detects laughter DIRECTLY from audio
- Trained on Google AudioSet (2M+ segments)
- Detects SOUND of laughter → language-agnostic → multilingual!
- Solves all our labeling problems

**Pipeline:**
1. Split each video into 10-second clips
2. Run AST model on each clip
3. Extract laughter probability
4. Use threshold to create labels

**Runtime:** ~1-2 hours on Colab T4 GPU

In [ ]:
# 1. Setup
!pip install transformers torch librosa tqdm 2>&1 | tail -3

from google.colab import drive
drive.mount('/content/drive')

import os, glob, time
import numpy as np
import torch
import librosa
from tqdm import tqdm
from transformers import pipeline

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

for BASE in ['/content/drive/My Drive/chuckle_net', '/content/drive/Shareddrives/chuckle_net']:
    if os.path.exists(BASE): break
AUDIO_DIR = f'{BASE}/audio'

In [ ]:
# 2. Load AST model
print('Loading AST model...')
classifier = pipeline(
    'audio-classification',
    model='MIT/ast-finetuned-audioset-10-10-0.4593',
    device=0 if device == 'cuda' else -1
)

# Test on a sample
audio_files = sorted(glob.glob(f'{AUDIO_DIR}/*.m4a'))
print(f'Found {len(audio_files)} audio files')

# Test inference on first 10 seconds of first file
y, sr = librosa.load(audio_files[0], sr=16000, duration=10)
result = classifier(y)

print(f'\nSample predictions for {os.path.basename(audio_files[0])}:')
for pred in result[:5]:
    print(f'  {pred["label"]}: {pred["score"]:.4f}')

# Check if laughter-related labels appear
laugh_labels = [r for r in result if 'laugh' in r['label'].lower() or 'giggl' in r['label'].lower()]
if laugh_labels:
    print(f'\n✅ Laughter detected!')
else:
    print(f'\n⚠️ No laughter in this segment (expected for non-laugh content)')

In [ ]:
# 3. Process all videos with AST
CLIP_LENGTH = 10  # seconds (AST expects 10s clips)
STRIDE = 5       # 50% overlap for better coverage
LAUGH_THRESHOLD = 0.3  # confidence threshold for laughter

all_results = []
t_start = time.time()

for idx, audio_path in enumerate(tqdm(audio_files, desc='Videos')):
    vid = os.path.basename(audio_path).replace('.m4a', '').replace('.wav', '')
    
    try:
        y, sr = librosa.load(audio_path, sr=16000)
    except:
        continue
    
    duration = len(y) / sr
    n_clips = int((duration - CLIP_LENGTH) / STRIDE) + 1
    
    for clip_idx in range(n_clips):
        start = clip_idx * STRIDE
        end = start + CLIP_LENGTH
        if end > duration:
            end = duration
            start = max(0, end - CLIP_LENGTH)
        
        clip = y[int(start * sr):int(end * sr)]
        if len(clip) < sr * 0.5:
            continue
        
        # Pad to 10 seconds if needed
        if len(clip) < CLIP_LENGTH * sr:
            clip = np.pad(clip, (0, CLIP_LENGTH * sr - len(clip)))
        
        result = classifier(clip)
        
        # Check for laughter-related labels
        laugh_score = 0
        for pred in result:
            label_lower = pred['label'].lower()
            if any(x in label_lower for x in ['laugh', 'giggl', 'chuckl', 'snicker']):
                laugh_score = max(laugh_score, pred['score'])
        
        all_results.append({
            'vid': vid,
            'start': start,
            'end': end,
            'laugh_score': laugh_score,
            'is_laugh': 1 if laugh_score > LAUGH_THRESHOLD else 0
        })
    
    if idx % 10 == 0:
        elapsed = time.time() - t_start
        rate = elapsed / (idx + 1)
        n_pos = sum(r['is_laugh'] for r in all_results)
        print(f'\n  [{idx+1}/{len(audio_files)}] {rate:.1f}s/vid, {n_pos} laugh clips ({100*n_pos/len(all_results):.1f}%)')

print(f'\n✅ Done in {(time.time()-t_start)/60:.1f} min')
print(f'Total clips: {len(all_results)}')
n_pos = sum(r['is_laugh'] for r in all_results)
print(f'Laughter clips: {n_pos} ({100*n_pos/len(all_results):.1f}%)')

In [ ]:
# 4. Extract prosody features + train
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import f1_score, precision_score, recall_score

# Convert to arrays
labels = np.array([r['is_laugh'] for r in all_results])
starts = np.array([r['start'] for r in all_results])
vids = np.array([r['vid'] for r in all_results])
scores = np.array([r['laugh_score'] for r in all_results])

print(f'Dataset: {len(labels)} clips, {labels.sum()} positive ({100*labels.mean():.1f}%)')
print(f'Videos: {len(set(vids))}')

# Video-level split
unique_vids = list(set(vids))
np.random.seed(42); np.random.shuffle(unique_vids)
n_test = max(1, int(len(unique_vids) * 0.2))
test_vids = set(unique_vids[:n_test])

test_mask = np.isin(vids, list(test_vids))

# Use AST scores as features (simple approach)
X = scores.reshape(-1, 1)
y = labels

X_train, X_test = X[~test_mask], X[test_mask]
y_train, y_test = y[~test_mask], y[test_mask]

print(f'Train: {len(X_train)} ({100*y_train.mean():.1f}% pos)')
print(f'Test:  {len(X_test)} ({100*y_test.mean():.1f}% pos)')

# The AST score IS the feature
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)
y_pred = lr.predict(X_test)
print(f'\n=== AST Score Threshold Model ===')
print(f'F1: {f1_score(y_test, y_pred):.4f}')
print(f'Precision: {precision_score(y_test, y_pred):.4f}')
print(f'Recall: {recall_score(y_test, y_pred):.4f}')

# Save results
import json
results_path = f'{BASE}/ast_labels_620.json'
with open(results_path, 'w') as f:
    json.dump(all_results, f)
print(f'\nSaved: {results_path}')

In [ ]:
# 5. Extract prosody features for each labeled clip
print('Extracting prosody features for each AST-labeled clip...')

all_feat = []
for r in tqdm(all_results, desc='Features'):
    audio_path = f'{AUDIO_DIR}/{r["vid"]}.m4a'
    if not os.path.exists(audio_path):
        all_feat.append([0]*23)
        continue
    try:
        y, sr = librosa.load(audio_path, sr=22050,
                            offset=r['start'], duration=r['end']-r['start'])
        rms = librosa.feature.rms(y=y, hop_length=512)[0]
        zcr = librosa.feature.zero_crossing_rate(y, hop_length=512)[0]
        sc = librosa.feature.spectral_centroid(y=y, sr=sr, hop_length=512)[0]
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13, hop_length=512)
        feat = [np.mean(rms), np.std(rms), np.max(rms), np.min(rms), np.mean(zcr),
                np.mean(sc), np.std(sc), np.max(sc), np.min(sc), np.mean(zcr > 0.1),
                np.mean(mfcc[0]), np.mean(mfcc[1]), np.mean(mfcc[2]), np.mean(mfcc[3]),
                np.mean(mfcc[4]), np.mean(mfcc[5]), np.mean(mfcc[6]), np.mean(mfcc[7]),
                np.mean(mfcc[8]), np.mean(mfcc[9]), np.mean(mfcc[10]), np.mean(mfcc[11]),
                np.mean(mfcc[12])]
        all_feat.append([float(f) for f in feat])
    except:
        all_feat.append([0]*23)

X_prosody = np.array(all_feat)

# Train on prosody features
lr2 = LogisticRegression(max_iter=1000)
lr2.fit(X_prosody[~test_mask], y[~test_mask])
y_pred2 = lr2.predict(X_prosody[test_mask])
print(f'\n=== Prosody Features (23-dim) with AST Labels ===')
print(f'F1: {f1_score(y[test_mask], y_pred2):.4f}')
print(f'Precision: {precision_score(y[test_mask], y_pred2):.4f}')
print(f'Recall: {recall_score(y[test_mask], y_pred2):.4f}')

# Save combined dataset
np.savez_compressed(f'{BASE}/ast_prosody_620.npz',
    features=X_prosody, labels=y, vids=vids, ast_scores=scores)
print(f'\n✅ Saved: {BASE}/ast_prosody_620.npz')